# Silver Layer — omr

## READINESS: partly profiled, rules drafted, awaiting remaining profiling and team approval

The key, missingness, date-range and `result_name` counts recorded in **Profiling Findings**
come from the run of 2026-10-19/20 (saved outputs of the previous version of this notebook).
The previous `result_value` shape check is **not reliable**: it classed every Blood Pressure
value (e.g. `110/65`) as numeric because it relied on `cast("double")`, so the value-format
checks are rewritten with explicit patterns and are *not yet profiled*. Until the
`profiling_approved` widget is set to `true`, the publish step raises an error and nothing
is written. The dependency-ordered runner excludes this notebook by default.

## Source
`<source_schema>.omr` (default `cdr_mimiciv.bronze.omr`)

## Target
`<target_schema>.omr` (default `cdr_mimiciv.silver.omr`), published with CREATE OR
REPLACE semantics. No tables are dropped.

## Grain (official MIMIC-IV hosp documentation, `omr`)
The Online Medical Record (OMR) table stores miscellaneous information documented in the
electronic health record; it is a useful source of outpatient measurements such as blood
pressure, weight, height and body mass index. Documented fields: `subject_id`, `chartdate`
(the date the observation was recorded; no time), `seq_num` (a monotonically increasing
integer which uniquely distinguishes results of the same type recorded on the same day,
ordered chronologically), `result_name` (a human-interpretable description of the
observation) and `result_value` (text; for Blood Pressure the recorded `systolic/diastolic`,
e.g. `120/80`). All five columns are documented NOT NULL. There is no row identifier; by
the `seq_num` definition (`subject_id`, `chartdate`, `result_name`, `seq_num`) identifies a
row. The documentation lists 13 `result_name` values (v2.2) with example values, including
eGFR `>60`, Weight `150.00` and Height `64.50`; it does not state units beyond those written
in the names. OMR has no `hadm_id`.

## Silver transformation (derived from the profiling findings)
Principle, as in admissions and labevents: preserve source value, identify anomaly, add a
derived column or DQ flag, let downstream users decide.
- Select every Bronze column unchanged, in its Bronze type and order. `result_name` and
  `result_value` are never overwritten.
- Add derived columns (our engineering choice), from an explicit mapping of the 13 profiled
  `result_name` values:
  - `result_measure`: Blood Pressure, Weight, Height, BMI or eGFR.
  - `bp_position`: Lying, Sitting, Standing, Standing (1 min), Standing (3 mins); NULL for
    plain Blood Pressure and other measures.
  - `result_unit`: the unit written in the name (`lbs`, `inches`, `kg/m2`); NULL where the name
    states none (Weight, Height, BMI, Blood Pressure, eGFR). A unit is not inferred.
  - `result_value_num`: `result_value` as a double where it is plain numeric text.
  - `bp_systolic`, `bp_diastolic`: the two parts of a `systolic/diastolic` value for Blood
    Pressure measures.
  - `value_comparator`, `value_censored_num`: for censored values such as eGFR `>60`
    (same names and meaning as labevents).
- Add DQ flags (our engineering choice), non-null booleans:
  - `dq_result_name_unmapped`: `result_name` not in the mapping (0 at last profile; catches
    new names in a later release).
  - `dq_result_value_unparsed`: `result_value` matches none of the numeric, ratio or censored
    patterns for its measure. Count *not yet profiled*.
- Add `_silver_processed_at`.
- No deduplication, no NULL filling, no unit conversion (lbs to kg, inches to cm), no merging
  of unit-less names into their unit-bearing twins, no plausibility capping, no row removal.
  Unit conversion, BMI recomputation, plausibility limits and "latest weight before
  admission" style features are Gold choices.

## Open profiling questions
1. `result_value` formats per `result_name` with explicit patterns (numeric, ratio, censored,
   other), and examples of "other".
2. Range of the parsed numbers per measure (min, percentiles, max): inspect, do not remove.
3. `seq_num` starts at 1 and has no gaps within (`subject_id`, `chartdate`, `result_name`)?

## Cost note
About 7.8 million rows. Profiling uses one aggregate pass plus grouped scans; no `exceptAll`
is used; the published table is checked with one aggregate pass.

In [0]:
import re

dbutils.widgets.text("source_schema", "cdr_mimiciv.bronze")
dbutils.widgets.text("target_schema", "cdr_mimiciv.silver")
dbutils.widgets.text("enforce_profiled_counts", "true")
dbutils.widgets.text("profiling_approved", "true")

BRONZE = dbutils.widgets.get("source_schema").strip()
SILVER = dbutils.widgets.get("target_schema").strip()
ENFORCE_PROFILED_COUNTS = dbutils.widgets.get("enforce_profiled_counts").strip().lower() == "true"
PROFILING_APPROVED = dbutils.widgets.get("profiling_approved").strip().lower() == "true"

for _schema in (BRONZE, SILVER):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", _schema):
        raise ValueError(f"Schema must be catalog.schema: {_schema}")
if BRONZE == SILVER:
    raise ValueError("source_schema and target_schema must differ")

from pyspark.sql import functions as F

SOURCE_TABLE = f"{BRONZE}.omr"
TARGET_TABLE = f"{SILVER}.omr"

DOCUMENTED_COLUMNS = ["subject_id", "chartdate", "seq_num", "result_name", "result_value"]
CANDIDATE_KEY = ["subject_id", "chartdate", "seq_num", "result_name"]

# explicit mapping of the profiled result_name values: (measure, bp_position, unit stated in the name)
RESULT_NAME_MAP = {
    "Blood Pressure":                   ("Blood Pressure", None, None),
    "Blood Pressure Lying":             ("Blood Pressure", "Lying", None),
    "Blood Pressure Sitting":           ("Blood Pressure", "Sitting", None),
    "Blood Pressure Standing":          ("Blood Pressure", "Standing", None),
    "Blood Pressure Standing (1 min)":  ("Blood Pressure", "Standing (1 min)", None),
    "Blood Pressure Standing (3 mins)": ("Blood Pressure", "Standing (3 mins)", None),
    "Weight (Lbs)":                     ("Weight", None, "lbs"),
    "Weight":                           ("Weight", None, None),
    "Height (Inches)":                  ("Height", None, "inches"),
    "Height":                           ("Height", None, None),
    "BMI (kg/m2)":                      ("BMI", None, "kg/m2"),
    "BMI":                              ("BMI", None, None),
    "eGFR":                             ("eGFR", None, None),
}

NUMERIC_PATTERN = r"^\s*-?[0-9]+(\.[0-9]+)?\s*$"
RATIO_PATTERN = r"^\s*([0-9]+(\.[0-9]+)?)\s*/\s*([0-9]+(\.[0-9]+)?)\s*$"
CENSORED_PATTERN = r"^\s*(<=|>=|<|>)\s*(-?[0-9]+(\.[0-9]+)?)\s*$"

In [0]:
omr_bronze = spark.table(SOURCE_TABLE)
SOURCE_COLUMNS = omr_bronze.columns

omr_bronze.printSchema()
print("Documented columns missing from Bronze:", [c for c in DOCUMENTED_COLUMNS if c not in SOURCE_COLUMNS])
print("Bronze columns not in documentation:", [c for c in SOURCE_COLUMNS if c not in DOCUMENTED_COLUMNS])
display(omr_bronze.limit(20))

root
 |-- subject_id: integer (nullable = true)
 |-- chartdate: date (nullable = true)
 |-- seq_num: integer (nullable = true)
 |-- result_name: string (nullable = true)
 |-- result_value: string (nullable = true)

Documented columns missing from Bronze: []
Bronze columns not in documentation: []


subject_id,chartdate,seq_num,result_name,result_value
10000032,2180-04-27,1,Blood Pressure,110/65
10000032,2180-04-27,1,Weight (Lbs),94
10000032,2180-05-07,1,BMI (kg/m2),18.0
10000032,2180-05-07,1,Height (Inches),60
10000032,2180-05-07,1,Weight (Lbs),92.15
10000032,2180-05-07,2,Weight (Lbs),92.15
10000032,2180-05-07,3,Weight (Lbs),92.15
10000032,2180-05-07,4,Weight (Lbs),92.15
10000032,2180-05-07,5,Weight (Lbs),92.15
10000032,2180-05-07,6,Weight (Lbs),92.15


In [0]:
# single-pass profile: row count, missingness of every column, date range
def _cnt(cond):
    return F.sum(F.when(cond, 1).otherwise(0))

omr_profile = omr_bronze.agg(
    F.count("*").alias("total_rows"),
    *[_cnt(F.col(c).isNull()).alias(f"null_{c}") for c in SOURCE_COLUMNS],
    _cnt(F.trim(F.col("result_value")) == "").alias("empty_result_value"),
    _cnt(F.col("seq_num") < 1).alias("seq_num_below_1"),
    F.min("chartdate").alias("min_chartdate"),
    F.max("chartdate").alias("max_chartdate"),
).first().asDict()
for k, v in omr_profile.items():
    print(f"{k}: {v}")

total_rows: 7753027
null_subject_id: 0
null_chartdate: 0
null_seq_num: 0
null_result_name: 0
null_result_value: 0
empty_result_value: 0
seq_num_below_1: 0
min_chartdate: 2105-01-19
max_chartdate: 2215-04-14


In [0]:
# grain: candidate natural key and exact duplicates
key_groups = omr_bronze.groupBy(*CANDIDATE_KEY).count()
print("Duplicate candidate-key groups:", key_groups.filter(F.col("count") > 1).count())

exact_duplicate_groups = omr_bronze.groupBy(*SOURCE_COLUMNS).count().filter(F.col("count") > 1)
print("Exact duplicate row groups:", exact_duplicate_groups.count())

# seq_num continuity within (subject_id, chartdate, result_name): starts at 1, no gaps
seq_groups = (
    omr_bronze.groupBy("subject_id", "chartdate", "result_name")
    .agg(F.min("seq_num").alias("min_seq"), F.max("seq_num").alias("max_seq"), F.count("*").alias("rows"))
)
display(
    seq_groups.agg(
        F.count("*").alias("groups"),
        _cnt(F.col("min_seq") != 1).alias("groups_not_starting_at_1"),
        _cnt(F.col("max_seq") != F.col("min_seq") + F.col("rows") - 1).alias("groups_with_gaps"),
    )
)

Duplicate candidate-key groups: 0
Exact duplicate row groups: 0


groups,groups_not_starting_at_1,groups_with_gaps
7464112,2667,174


In [0]:
# categorical values: result_name, with case/whitespace variants
display(omr_bronze.groupBy("result_name").count().orderBy(F.desc("count")))

display(
    omr_bronze.select(
        F.countDistinct("result_name").alias("distinct_result_name"),
        F.countDistinct(F.lower(F.trim("result_name"))).alias("distinct_result_name_normalised")
    )
)

result_name,count
Blood Pressure,2827801
Weight (Lbs),2145353
BMI (kg/m2),1901496
Height (Inches),814964
BMI,29021
Weight,15108
Blood Pressure Sitting,5757
Blood Pressure Standing (1 min),4704
Blood Pressure Lying,4359
Height,2365


distinct_result_name,distinct_result_name_normalised
13,13


In [0]:
# result_value shape per result_name, by explicit pattern (inspect, do not remove)
value_shape = (
    F.when(F.col("result_value").rlike(NUMERIC_PATTERN), "numeric")
     .when(F.col("result_value").rlike(RATIO_PATTERN), "ratio")
     .when(F.col("result_value").rlike(CENSORED_PATTERN), "censored")
     .otherwise("other")
)
display(
    omr_bronze.groupBy("result_name", value_shape.alias("value_shape"))
    .count()
    .orderBy("result_name", F.desc("count"))
)

display(
    omr_bronze.where(value_shape == "other")
    .groupBy("result_name", "result_value")
    .count()
    .orderBy(F.desc("count"))
    .limit(100)
)

result_name,value_shape,count
BMI,numeric,29021
BMI (kg/m2),numeric,1901496
Blood Pressure,ratio,2827801
Blood Pressure Lying,ratio,4359
Blood Pressure Sitting,ratio,5757
Blood Pressure Standing,ratio,565
Blood Pressure Standing (1 min),ratio,4704
Blood Pressure Standing (3 mins),ratio,1255
Height,numeric,2323
Height,other,42


result_name,result_value,count
Height (Inches),63.,20
Height (Inches),62.,18
Height (Inches),66.,16
Height (Inches),65.,15
Height (Inches),61.,15
Weight (Lbs),173.,15
Height (Inches),64.,14
Weight (Lbs),160.,14
Weight (Lbs),182.,13
Height (Inches),69.,13


In [0]:
# relationship (observational only)
patients_table = f"{SILVER}.patients"
assert spark.catalog.tableExists(patients_table), f"Missing reference: {patients_table}"

patients_ref = spark.table(patients_table).select("subject_id").distinct()
print(
    "distinct subject_id not found in patients:",
    omr_bronze.select("subject_id").distinct().join(patients_ref, "subject_id", "left_anti").count()
)

distinct subject_id not found in patients: 0


## Profiling Findings

Recorded from the profiling run of 2026-10-19/20 on `cdr_mimiciv.bronze.omr` (saved outputs
of the previous notebook version). Rows marked *not yet profiled* must be filled from the
cells above before `profiling_approved` is set.
[DOC] = stated in official MIMIC documentation; [OURS] = our engineering choice.

| Check | Result | Silver treatment |
|---|---:|---|
| Rows | 7,753,027 | Row count preserved |
| Candidate key (`subject_id`, `chartdate`, `seq_num`, `result_name`) | 0 duplicate groups | Unique; [DOC] implied by `seq_num` definition; asserted after publish |
| Exact duplicate rows | 0 groups | None |
| NULL in every column / empty `result_value` | 0 / 0 | None; [DOC] all NOT NULL |
| `seq_num` continuity | *not yet profiled* | Report only |
| `chartdate` range | 2105-01-19 to 2215-04-14 | None; [DOC] dates shifted |
| `result_name` values | 13 names, no case/whitespace variants: Blood Pressure 2,827,801; Weight (Lbs) 2,145,353; BMI (kg/m2) 1,901,496; Height (Inches) 814,964; BMI 29,021; Weight 15,108; BP Sitting 5,757; BP Standing (1 min) 4,704; BP Lying 4,359; Height 2,365; BP Standing (3 mins) 1,255; BP Standing 565; eGFR 279 | Kept; mapped to `result_measure` / `bp_position` / `result_unit` [OURS]; unmapped names flagged `dq_result_name_unmapped` |
| Unit-less names (Weight, Height, BMI) | 46,494 rows | `result_unit` NULL; not merged with the unit-bearing names [OURS] |
| `result_value` formats | *not yet profiled* (previous shape check unreliable) | Parsed into `result_value_num`, `bp_systolic` / `bp_diastolic`, `value_comparator` / `value_censored_num`; rest flagged `dq_result_value_unparsed` [OURS] |
| Parsed value ranges per measure | *not yet profiled* | Report only; plausibility limits are a Gold choice |
| `subject_id` not in patients | 0 distinct | None |

In [0]:
# Silver transformation: all source columns unchanged + derived columns + DQ flags
# (our engineering choices, see Profiling Findings); no rows removed, no source value changed
name_map = spark.createDataFrame(
    [(k, *v) for k, v in RESULT_NAME_MAP.items()],
    "result_name string, result_measure string, bp_position string, result_unit string",
)

value_trim = F.trim(F.col("result_value"))
is_bp = F.col("result_measure") == "Blood Pressure"
is_numeric = value_trim.rlike(NUMERIC_PATTERN)
is_ratio = is_bp & value_trim.rlike(RATIO_PATTERN)
is_censored = value_trim.rlike(CENSORED_PATTERN)

omr_silver = (
    omr_bronze.select(*SOURCE_COLUMNS)
    .join(F.broadcast(name_map), "result_name", "left")
    .select(
        *SOURCE_COLUMNS,
        "result_measure",
        "bp_position",
        "result_unit",
        F.when(is_numeric, value_trim.cast("double")).alias("result_value_num"),
        F.when(is_ratio, F.regexp_extract(value_trim, RATIO_PATTERN, 1).cast("double")).alias("bp_systolic"),
        F.when(is_ratio, F.regexp_extract(value_trim, RATIO_PATTERN, 3).cast("double")).alias("bp_diastolic"),
        F.when(is_censored, F.regexp_extract(value_trim, CENSORED_PATTERN, 1)).alias("value_comparator"),
        F.when(is_censored, F.regexp_extract(value_trim, CENSORED_PATTERN, 2).cast("double"))
            .alias("value_censored_num"),
        F.col("result_measure").isNull().alias("dq_result_name_unmapped"),
        F.coalesce(~(is_numeric | is_ratio | is_censored), F.lit(True)).alias("dq_result_value_unparsed"),
        F.current_timestamp().alias("_silver_processed_at"),
    )
)

DQ_FLAGS = ["dq_result_name_unmapped", "dq_result_value_unparsed"]

expected_schema = {f.name: f.dataType.simpleString() for f in omr_bronze.schema.fields}
expected_schema.update({
    "result_measure": "string",
    "bp_position": "string",
    "result_unit": "string",
    "result_value_num": "double",
    "bp_systolic": "double",
    "bp_diastolic": "double",
    "value_comparator": "string",
    "value_censored_num": "double",
    "dq_result_name_unmapped": "boolean",
    "dq_result_value_unparsed": "boolean",
    "_silver_processed_at": "timestamp",
})

# profiled expected counts (2026-10-19/20 run); add dq_result_value_unparsed once profiled
PROFILED_COUNTS = {
    "silver_rows": 7753027,
    "dq_result_name_unmapped": 0,
}

omr_silver.printSchema()

root
 |-- subject_id: integer (nullable = true)
 |-- chartdate: date (nullable = true)
 |-- seq_num: integer (nullable = true)
 |-- result_name: string (nullable = true)
 |-- result_value: string (nullable = true)
 |-- result_measure: string (nullable = true)
 |-- bp_position: string (nullable = true)
 |-- result_unit: string (nullable = true)
 |-- result_value_num: double (nullable = true)
 |-- bp_systolic: double (nullable = true)
 |-- bp_diastolic: double (nullable = true)
 |-- value_comparator: string (nullable = true)
 |-- value_censored_num: double (nullable = true)
 |-- dq_result_name_unmapped: boolean (nullable = false)
 |-- dq_result_value_unparsed: boolean (nullable = false)
 |-- _silver_processed_at: timestamp (nullable = false)



In [0]:
# publish gate: nothing below runs until profiling has been reviewed and approved
if not PROFILING_APPROVED:
    raise RuntimeError(
        "omr is not approved for publication: run and review the profiling cells, fill the "
        "'not yet profiled' findings and any justified rules, then set widget profiling_approved=true."
    )

# cheap pre-publish validation (no data scan)
pre_schema = {f.name: f.dataType.simpleString() for f in omr_silver.schema.fields}
assert list(pre_schema.items()) == list(expected_schema.items()), "Unexpected Silver schema"

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

(
    omr_silver.writeTo(TARGET_TABLE)
    .using("delta")
    .createOrReplace()
)

print(f"Published: {TARGET_TABLE}")

Published: cdr_mimiciv.silver.omr


In [0]:
# post-publish validation: one aggregate pass over Silver, one count of Bronze (no exceptAll)
published = spark.table(TARGET_TABLE)

post = published.agg(
    F.count("*").alias("silver_rows"),
    F.countDistinct(*CANDIDATE_KEY).alias("distinct_candidate_key"),
    _cnt(F.col("_silver_processed_at").isNull()).alias("null_processed_at"),
    _cnt(F.greatest(*[F.col(c).isNull() for c in DQ_FLAGS])).alias("null_dq_flags"),
    *[_cnt(F.col(c)).alias(c) for c in DQ_FLAGS],
    _cnt(F.col("result_value_num").isNotNull() & F.col("value_censored_num").isNotNull())
        .alias("numeric_and_censored"),
    _cnt(F.col("bp_systolic").isNotNull() & (F.col("result_measure") != "Blood Pressure"))
        .alias("bp_parts_outside_bp"),
).first().asDict()
bronze_rows = omr_bronze.count()

published_schema = {f.name: f.dataType.simpleString() for f in published.schema.fields}

print("Bronze rows:", bronze_rows)
print("Published:", post)

assert post["silver_rows"] == bronze_rows, "Silver row count differs from Bronze"
# the candidate key was profiled unique ([DOC] seq_num distinguishes same-day results)
assert post["distinct_candidate_key"] == post["silver_rows"], "Candidate key is not unique"
assert post["null_processed_at"] == 0
assert post["null_dq_flags"] == 0, "DQ flags must be non-null booleans"
assert post["numeric_and_censored"] == 0, "A value parsed as both numeric and censored"
assert post["bp_parts_outside_bp"] == 0, "Blood pressure parts set for a non-BP measure"
assert list(published_schema.items()) == list(expected_schema.items()), \
    f"Schema mismatch: {published_schema}"

if ENFORCE_PROFILED_COUNTS:
    for name, expected in PROFILED_COUNTS.items():
        assert post[name] == expected, f"{name}: expected {expected}, got {post[name]}"

dbutils.notebook.exit("omr:OK")